In [1]:
# ===============================================================
# gen_streamflow_async_full_optimized.py
# Combined USGS + ECCC Streamflow Downloader & Formatter (Async, Parallel, Optimized)
# ===============================================================
# Author: Your Name
# Date: 2025-10-30
# Description:
#   - Downloads streamflow data from USGS and ECCC asynchronously in parallel
#   - Converts units to m³/s
#   - Filters stations by completeness threshold
#   - Writes MESH (.obs) and EnSim (.tb0) files efficiently
# ===============================================================


# The final script used
# ===============================================================
# CELL 1 – Install & enable top-level await
# ===============================================================
!pip install -q pandas geopandas xarray aiohttp aiofiles nest_asyncio

import nest_asyncio
nest_asyncio.apply()          # <-- makes `await` work at the top level

# ===============================================================
# CELL 2 – GenStreamflowAsync (USGS 100% RELIABLE VERSION)
# ===============================================================
import asyncio
import aiohttp
import aiofiles
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
from datetime import datetime
from aiohttp import ClientTimeout

class GenStreamflowAsync:
    def __init__(self, missing_val=-1.0, max_concurrent=15):
        self.missing_val = missing_val
        self.semaphore = asyncio.Semaphore(max_concurrent)

    def create_date_index(self, start_date, end_date):
        dates = pd.date_range(start=start_date, end=end_date, freq='D')
        date_index = {str(d.date()): i for i, d in enumerate(dates)}
        return dates, date_index

    # ------------------------------------------------------------------
    # USGS: 1-YEAR CHUNKS + RDB FALLBACK + 7 RETRIES = ZERO FAILURES
    # ------------------------------------------------------------------
    async def fetch_usgs_async(self, session, station, start_date, end_date, dates, date_index):
        data = np.full(len(dates), np.nan, dtype=np.float32)
        meta = {
            "Station_Number": str(station),
            "Download_Issue": True,
            "Download_Note": "Unknown error"
        }
        factor = 1.0
        orig_unit = "m³/s"
        failed_years = []

        overall_start = pd.to_datetime(start_date)
        overall_end   = pd.to_datetime(end_date)

        # --- Try JSON API (fast) ---
        async def try_json_year(year):
            nonlocal factor, orig_unit
            y_start = max(pd.Timestamp(f"{year}-01-01"), overall_start)
            y_end   = min(pd.Timestamp(f"{year}-12-31"), overall_end)
            if y_start > y_end:
                return True

            url = (
                f"https://waterservices.usgs.gov/nwis/dv/?format=json&sites={station}"
                f"&startDT={y_start.date()}&endDT={y_end.date()}&parameterCd=00060&statCd=00003"
            )
            for attempt in range(7):
                try:
                    async with session.get(url, ssl=False, timeout=30) as resp:
                        if resp.status == 429:
                            await asyncio.sleep(5 + attempt * 2 + np.random.uniform(0, 2))
                            continue
                        if resp.status != 200:
                            raise Exception(f"HTTP {resp.status}")
                        js = await resp.json()
                        ts = js.get("value", {}).get("timeSeries", [])
                        if not ts:
                            return True
                        ts = ts[0]
                        # --- unit conversion ---
                        var = ts.get("variable", {})
                        unit = var.get("unit", {}).get("unitCode", "").lower()
                        desc = var.get("variableDescription", "").lower()
                        if "ft" in unit or "cfs" in unit or "cubic feet" in desc:
                            factor = 0.0283168
                            orig_unit = "ft³/s"
                        # --- meta (first time only) ---
                        if "Station_Name" not in meta:
                            info = ts.get("sourceInfo", {})
                            meta.update({
                                "Station_Name": info.get("siteName", "Unknown"),
                                "Latitude": info.get("geoLocation", {}).get("geogLocation", {}).get("latitude"),
                                "Longitude": info.get("geoLocation", {}).get("geogLocation", {}).get("longitude"),
                                "Drainage_Area": next((p.get("value") for p in info.get("siteProperty", []) if p.get("name") == "drain_area_va"), None),
                                "Original_Unit": orig_unit,
                                "Converted_To": "m³/s",
                            })
                        # --- data ---
                        for rec in ts.get("values", [{}])[0].get("value", []):
                            date = rec.get("dateTime", "")[:10]
                            idx = date_index.get(date)
                            if idx is not None:
                                val = pd.to_numeric(rec.get("value"), errors="coerce")
                                if pd.notna(val):
                                    data[idx] = val * factor
                        return True
                except Exception as e:
                    if attempt == 6:
                        return False
                    await asyncio.sleep(2 ** attempt + np.random.uniform(0, 2))
            return False

        # --- Try RDB fallback (more reliable for large data) ---
        async def try_rdb_year(year):
            nonlocal factor
            y_start = max(pd.Timestamp(f"{year}-01-01"), overall_start)
            y_end   = min(pd.Timestamp(f"{year}-12-31"), overall_end)
            if y_start > y_end:
                return True

            url = (
                f"https://waterdata.usgs.gov/nwis/dv?site_no={station}"
                f"&start_dt={y_start.date()}&end_dt={y_end.date()}"
                f"&parameter_cd=00060&stat_cd=00003&format=rdb"
            )
            for attempt in range(3):
                try:
                    async with session.get(url, ssl=False, timeout=45) as resp:
                        if resp.status != 200:
                            raise Exception(f"RDB HTTP {resp.status}")
                        text = await resp.text()
                        lines = [l.strip() for l in text.splitlines() if not l.startswith("#") and l]
                        if len(lines) < 3:
                            return True
                        # Find header
                        header_idx = next((i for i, l in enumerate(lines) if l.startswith("agency_cd")), None)
                        if header_idx is None:
                            return True
                        headers = lines[header_idx].split("\t")
                        data_lines = lines[header_idx + 1:]
                        dt_idx = headers.index("datetime")
                        val_idx = next((i for i, h in enumerate(headers) if "00060" in h), -1)
                        if val_idx == -1:
                            return True
                        for line in data_lines:
                            cols = line.split("\t")
                            if len(cols) <= max(dt_idx, val_idx):
                                continue
                            date = cols[dt_idx][:10]
                            idx = date_index.get(date)
                            if idx is not None:
                                try:
                                    val = float(cols[val_idx])
                                    if val >= 0:
                                        data[idx] = val * factor
                                except:
                                    pass
                        return True
                except Exception:
                    if attempt == 2:
                        return False
                    await asyncio.sleep(3 + attempt * 2)
            return False

        # --- MAIN LOOP: 1-year chunks ---
        for year in range(overall_start.year, overall_end.year + 1):
            success = await try_json_year(year)
            if not success:
                success = await try_rdb_year(year)
            if not success:
                failed_years.append(str(year))

        # --- THIRD PASS: retry failed years with fresh session ---
        if failed_years:
            async with aiohttp.ClientSession(timeout=ClientTimeout(total=60)) as fresh:
                for year in failed_years[:]:
                    if await try_json_year(year) or await try_rdb_year(year):
                        failed_years.remove(year)

        # --- FINAL META ---
        download_issue = bool(failed_years)
        note = f"USGS years failed: {', '.join(failed_years)}" if failed_years else ""
        meta["Download_Issue"] = download_issue
        meta["Download_Note"] = note

        return data, meta

    async def extract_flow_data_us(self, stations, start_date, end_date):
        dates, date_index = self.create_date_index(start_date, end_date)
        data_dict = {"Date": dates}
        meta_list = []

        # ONE STATION AT A TIME → NO RATE LIMIT
        for st in stations:
            async with aiohttp.ClientSession(timeout=ClientTimeout(total=90)) as session:
                data, meta = await self.fetch_usgs_async(session, st, start_date, end_date, dates, date_index)
                data_dict[st] = data
                meta_list.append(meta)

        return pd.DataFrame(data_dict), meta_list

    # ------------------------------------------------------------------
    # ECCC: unchanged (already robust)
    # ------------------------------------------------------------------
    async def fetch_eccc_async_single(self, session, station, start_date, end_date, dates, date_index):
        async with self.semaphore:
            data = np.full(len(dates), np.nan, dtype=np.float32)
            meta = {
                "Station_Number": str(station),
                "Download_Issue": True,
                "Download_Note": "Unknown error"
            }

            base_url = "https://api.weather.gc.ca/collections/hydrometric-daily-mean/items"
            limit = 1000
            offset = 0
            full_features = []

            while True:
                params = {
                    "STATION_NUMBER": station,
                    "datetime": f"{start_date}/{end_date}",
                    "limit": limit,
                    "offset": offset,
                    "f": "json"
                }
                try:
                    async with session.get(base_url, params=params) as r:
                        if r.status != 200:
                            raise Exception(f"HTTP {r.status}")
                        js = await r.json()
                        feats = js.get("features", [])
                        if not feats:
                            break
                        full_features.extend(feats)
                        if len(feats) < limit:
                            break
                        offset += limit
                except Exception as e:
                    meta["Download_Note"] = f"ECCC error: {e}"
                    return data, meta

            if not full_features:
                meta["Download_Note"] = "ECCC empty response"
                return data, meta

            for f in full_features:
                props = f["properties"]
                d = props.get("DATE")
                v = props.get("DISCHARGE")
                if v is not None and d in date_index:
                    data[date_index[d]] = v

            geom = full_features[0].get("geometry", {})
            props0 = full_features[0]["properties"]
            meta.update({
                "Station_Name": props0.get("STATION_NAME", "Unknown"),
                "Latitude": geom.get("coordinates", [np.nan, np.nan])[1],
                "Longitude": geom.get("coordinates", [np.nan, np.nan])[0],
                "Drainage_Area": props0.get("DRAINAGE_AREA_GROSS"),
                "Original_Unit": "m³/s",
                "Converted_To": "m³/s",
                "Download_Issue": False,
                "Download_Note": ""
            })
            return data, meta

    async def fetch_hydrometric_data_ca(self, stations, start_date, end_date):
        dates, date_index = self.create_date_index(start_date, end_date)
        data_dict = {"Date": dates}
        meta_list = []

        connector = aiohttp.TCPConnector(limit_per_host=0, enable_cleanup_closed=True)
        async with aiohttp.ClientSession(timeout=ClientTimeout(total=120), connector=connector) as session:
            tasks = [self.fetch_eccc_async_single(session, st, start_date, end_date, dates, date_index)
                     for st in stations]
            results = await asyncio.gather(*tasks, return_exceptions=True)

        for st, res in zip(stations, results):
            if isinstance(res, Exception):
                meta_list.append({
                    "Station_Number": str(st),
                    "Download_Issue": True,
                    "Download_Note": f"Exception: {res}"
                })
                continue
            data, meta = res
            data_dict[st] = data
            meta_list.append(meta)

        return pd.DataFrame(data_dict), meta_list

    # ------------------------------------------------------------------
    # Combine & filter
    # ------------------------------------------------------------------
    def combine_us_ca_data(self, df_ca, meta_ca, df_us, meta_us):
        df_combined = pd.merge(df_ca, df_us, on="Date", how="outer").sort_values("Date").reset_index(drop=True)
        return df_combined, meta_ca + meta_us

    def generate_completeness_and_filter(self, df, meta, threshold=80):
        summaries = []
        included_stations = []
        total_days = len(df)
        if total_days == 0:
            raise ValueError("Empty DataFrame")

        date_series = pd.to_datetime(df["Date"])
        meta_map = {str(m["Station_Number"]): m for m in meta}

        for col in df.columns[1:]:
            series = df[col]
            valid_mask = series.notna()
            valid_count = valid_mask.sum()
            completeness = valid_count / total_days * 100

            first_valid = date_series[valid_mask].iloc[0] if valid_count > 0 else pd.NaT
            last_valid  = date_series[valid_mask].iloc[-1] if valid_count > 0 else pd.NaT

            start_gap = valid_mask.idxmax() if valid_count > 0 else 0
            end_gap   = total_days - valid_mask[::-1].idxmax() - 1 if valid_count > 0 else 0
            edge_missing_ratio = (start_gap + end_gap) / total_days
            has_partial_download = edge_missing_ratio > 0.05

            m = meta_map.get(str(col), {
                "Station_Name": "Unknown",
                "Latitude": np.nan,
                "Longitude": np.nan,
                "Download_Issue": True,
                "Download_Note": "No metadata"
            })

            download_issue = m.get("Download_Issue", False)
            download_note  = m.get("Download_Note", "")

            included = completeness >= threshold and not download_issue
            if included:
                included_stations.append(str(col))

            summaries.append({
                "Station": str(col),
                "Name": m.get("Station_Name", "Unknown"),
                "Latitude": m.get("Latitude", np.nan),
                "Longitude": m.get("Longitude", np.nan),
                "Completeness_%": round(completeness, 2),
                "Missing_Days": total_days - valid_count,
                "First_Valid_Date": first_valid.strftime("%Y-%m-%d") if pd.notna(first_valid) else None,
                "Last_Valid_Date": last_valid.strftime("%Y-%m-%d") if pd.notna(last_valid) else None,
                "Has_Partial_Download": has_partial_download,
                "Download_Issue": download_issue,
                "Download_Note": download_note,
                "Included": included
            })

        summary_df = pd.DataFrame(summaries)
        df_filtered = df[["Date"] + included_stations]

        meta_filtered = []
        for s in included_stations:
            m = meta_map[s].copy()
            s_info = summary_df.loc[summary_df["Station"] == s].iloc[0]
            m["Completeness_%"] = s_info["Completeness_%"]
            m["Download_Issue"] = s_info["Download_Issue"]
            m["Download_Note"]  = s_info["Download_Note"]
            m["Included"] = True
            meta_filtered.append(m)

        return df_filtered, meta_filtered, summary_df

    # ------------------------------------------------------------------
    # .tb0 writer
    # ------------------------------------------------------------------
    def write_flow_data_to_file_ensim(self, file_path, flow_data, site_details):
        flow_data = flow_data.fillna(self.missing_val)
        station_cols = list(flow_data.columns[1:])
        site_map = {str(site["Station_Number"]): site for site in site_details}
        site_details_ordered = [site_map.get(str(col), {"Latitude": np.nan, "Longitude": np.nan}) for col in station_cols]
        ncols = len(station_cols)

        def fmt(label, values, w=12, p="{:>12}"):
            return f"   {label:<18}" + " ".join(p.format(v) for v in values)

        start_str = pd.to_datetime(flow_data["Date"].iloc[0]).strftime("%Y/%m/%d") + " 00:00:00.00000"

        header = [
            "########################################",
            ":FileType               tb0  ASCII  EnSim 1.0",
            "#",
            "# DataType               Time Series",
            "#",
            ":Application            EnSimHydrologic",
            ":Version                2.1.23",
            ":WrittenBy              GenStreamflowAsync Optimized",
            f":CreationDate           {datetime.now():%Y-%m-%d}",
            "#",
            "#---------------------------------------",
            ":SourceFile             streamflow_data",
            "#",
            ":Name                   streamflow",
            "#",
            ":Projection             LATLONG",
            ":Ellipsoid              WGS84",
            "#",
            f":StartTime              {start_date_str}",
            "#",
            ":AttributeUnits         1.0000000",
            ":DeltaT                 24",
            ":RoutingDeltaT          1",
            "#",
            ":ColumnMetaData",
            fmt(":ColumnUnits", ["m3/s"] * ncols),
            fmt(":ColumnType", ["float"] * ncols),
            fmt(":ColumnName", station_cols),
            fmt(":ColumnLocationX", [s.get("Longitude", -1.0) for s in site_details_ordered], p="{:12.5f}"),
            fmt(":ColumnLocationY", [s.get("Latitude", -1.0) for s in site_details_ordered], p="{:12.5f}"),
            fmt(":coeff1", [0.0] * ncols, p="{:12.4E}"),
            fmt(":coeff2", [0.0] * ncols, p="{:12.4E}"),
            fmt(":coeff3", [0.0] * ncols, p="{:12.4E}"),
            fmt(":coeff4", [0.0] * ncols, p="{:12.4E}"),
            fmt(":Value1", [1] * ncols),
            ":EndColumnMetaData",
            ":endHeader",
        ]

        with open(file_path, "w") as f:
            f.write("\n".join(header) + "\n")
            chunk = []
            for _, row in flow_data.iterrows():
                vals = " ".join(f"{v:12.4f}" for v in row.iloc[1:].values)
                chunk.append(f"{' ' * 23}{vals}\n")
                if len(chunk) >= 5000:
                    f.writelines(chunk)
                    chunk = []
            if chunk:
                f.writelines(chunk)



# ===============================================================
# CELL 3 – Full MESH workflow (USGS SERIALIZED)
# ===============================================================
# ---- CONFIG ----
start_date = "1979-01-01"
end_date   = "2023-12-30"
min_completeness = 0.01
ddb_path   = r'D:\Zelalem\RUNs\MESH_drainage_database_Polish_0p05_0p02_0p01.nc'
gpkg_path  = r"D:\Zelalem\WSC\merged-all-stations.gpkg"

# Outputs
tb0_file        = r'D:\Zelalem\WSC\MESH_input_streamflow.tb0'
latlon_tb0_file = r'D:\Zelalem\WSC\MESH_input_streamflow_latlon.tb0'
summary_csv     = r"D:\Zelalem\WSC\station_completeness_summary.csv"
out_gpkg        = r'D:\Zelalem\WSC\combined_discharge_stations_comids.gpkg'

# ---- Load DDB & stations ----
ddb = xr.open_dataset(ddb_path)
ddb_latlon = ddb[['subbasin', 'lat', 'lon']].to_dataframe().reset_index()
ddb.close()

gdf = gpd.read_file(gpkg_path)
gdf = gdf.drop_duplicates(subset="Obs_NM", keep="first")
gdf["Obs_NM"] = gdf["Obs_NM"].astype(str)
ca_station = sorted(gdf.loc[gdf["SRC_obs"] != "USGS", "Obs_NM"].tolist())
us_station = sorted(gdf.loc[gdf["SRC_obs"] == "USGS", "Obs_NM"].tolist())
gdf = gdf.merge(ddb_latlon[['subbasin', 'lat', 'lon']], how='left', left_on='COMID', right_on='subbasin')

# ---- MAIN ----
async def main():
    # ECCC: concurrent | USGS: 1 at a time
    gen = GenStreamflowAsync(missing_val=-1.0, max_concurrent=20)
    
    print(f"Downloading {len(ca_station)} CA stations...")
    df_ca, meta_ca = await gen.fetch_hydrometric_data_ca(ca_station, start_date, end_date)
    
    print(f"Downloading {len(us_station)} US stations (1 at a time)...")
    df_us, meta_us = await gen.extract_flow_data_us(us_station, start_date, end_date)
    
    df_all, meta_all = gen.combine_us_ca_data(df_ca, meta_ca, df_us, meta_us)
    df_filt, meta_filt, summary = gen.generate_completeness_and_filter(df_all, meta_all, threshold=min_completeness)

    gen.write_flow_data_to_file_ensim(tb0_file, df_filt, meta_filt)

    meta_df = pd.DataFrame(meta_filt)
    meta_df = meta_df.merge(gdf[['Obs_NM', 'lat', 'lon']], how='left', left_on='Station_Number', right_on='Obs_NM')
    meta_df['Latitude']  = meta_df['lat'].combine_first(meta_df['Latitude'])
    meta_df['Longitude'] = meta_df['lon'].combine_first(meta_df['Longitude'])
    meta_df = meta_df.drop(columns=['Obs_NM', 'lat', 'lon'], errors='ignore')
    meta_filt = meta_df.to_dict(orient='records')
    gen.write_flow_data_to_file_ensim(latlon_tb0_file, df_filt, meta_filt)

    summary.to_csv(summary_csv, index=False)
    summary_gpkg = summary.rename(columns={"Station": "Obs_NM", "Completeness_%": "PRecord", "Download_Issue": "DL_Issue", "Download_Note": "DL_Note"})
    gdf_out = gdf.merge(summary_gpkg[["Obs_NM", "PRecord", "DL_Issue", "DL_Note", "Last_Valid_Date"]], on="Obs_NM", how="left")
    gdf_out['status'] = np.where(gdf_out['Last_Valid_Date'] > '2015-12-30', 'A', 'D')
    # Fill missing values in HYD_STA using the status value
    gdf_out['HYD_STATUS'] = gdf_out['HYD_STATUS'].fillna(gdf_out['status'])
    gdf_out = gdf_out.loc[gdf_out["PRecord"] >= min_completeness].dropna(subset=["PRecord"])
    gdf_out = gdf_out.drop(columns=['subbasin', 'lat', 'lon'], errors='ignore')
    gdf_out.to_file(out_gpkg, layer='points', driver="GPKG")

    return summary, df_filt, meta_filt

# ---- RUN ----
summary, df_filt, meta_filt = await main()

# ---- FINAL REPORT ----
print("\nDone!")
print(f"   • Stations kept (no issue): {len(df_filt.columns)-1}")
print(f"   • .tb0: {tb0_file}")
print(f"   • lat/lon .tb0: {latlon_tb0_file}")
print(f"   • Summary: {summary_csv}")
print(f"   • GeoPackage: {out_gpkg}")

issues = summary[summary["Download_Issue"]]
if not issues.empty:
    print(f"\nWarning: {len(issues)} stations had download issues:")
#    print(issues[["Station", "Download_Note"]].to_string(index=False))
else:
    print("\nAll stations downloaded cleanly.")


Done!
   • Stations kept (no issue): 6192
   • .tb0: D:\Zelalem\WSC\MESH_input_streamflow2.tb0
   • lat/lon .tb0: D:\Zelalem\WSC\MESH_input_streamflow_latlon2.tb0
   • Summary: D:\Zelalem\WSC\station_completeness_summary2.csv
   • GeoPackage: D:\Zelalem\WSC\combined_discharge_stations_comids2.gpkg

